# Metrics from the synthetic sample

This notebook runs on `data/sample_synthetic.csv`, which is **generated, not traded**.
It exists so the metric code in `src/` can be read and executed by anyone who clones
this repo, without any private data.

Do not quote a number from this notebook. The real aggregates are in
`data/public_monthly_stats.csv`.


In [ ]:
import csv, sys
from datetime import datetime, timezone
from decimal import Decimal
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src import metrics
from src.realized import RealizedEvent
from src.schema import Side

rows = list(csv.DictReader(open("../data/sample_synthetic.csv")))
print(len(rows), "SYNTHETIC rows")
rows[0]

## Load into the same typed record the real pipeline uses

In [ ]:
events = [
    RealizedEvent(
        ts=datetime.strptime(r["closed_at_utc"], "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc),
        symbol=r["symbol"], side=Side(r["side"]), qty=Decimal(r["qty"]),
        price=Decimal(r["exit_price"]), gross_pnl_usdt=Decimal(r["gross_pnl_usdt"]),
        exit_fee_usdt=Decimal(r["exit_fee_usdt"]), maker=r["role"] == "maker",
        source="synthetic",
    )
    for r in rows
]
total_fees = float(sum(e.exit_fee_usdt for e in events))
len(events)

## Summary

Note what is **not** here: nothing in units of R. `planned_risk_usdt` is populated in
this synthetic file but is empty in anything public, and the code returns `None`
rather than substituting a proxy denominator. An R computed off a guessed
denominator is worse than no R.

In [ ]:
s = metrics.summarize_realized(events, total_fees)
for f in s.__slots__:
    v = getattr(s, f)
    print(f"{f:24} {v:,.4f}" if isinstance(v, float) else f"{f:24} {v}")

## Drawdown on the cumulative net curve

In [ ]:
daily = {}
for e in events:
    d = e.ts.date()
    daily[d] = daily.get(d, 0.0) + float(e.gross_pnl_usdt) - float(e.exit_fee_usdt)
daily = sorted(daily.items())

cum = metrics.cumulative_daily(daily)
dd = metrics.max_drawdown(cum)
print(f"max drawdown {dd.max_dd_usdt:,.2f} USDT   peak {dd.peak_at} -> trough {dd.trough_at}")
print(f"recovered: {dd.recovered_at or 'not in window'}")

## Why a percent return is not printed here

`equity_curve` returns `None` without a starting capital, on purpose. A percentage
has no meaning without the equity it is a percentage of, and an order export
contains no deposits or withdrawals.

In [ ]:
print(metrics.equity_curve(daily, None))
eq = metrics.equity_curve(daily, 15_000)
print(f"at a stated 15,000 capital: index ends at {eq[-1][1]:.1f}")